[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/s3_raytracer/s3_raytracer.ipynb)

# Ray Tracing on the 3-Sphere

This example renders identical quadric ellipsoids arranged along a geodesic line of sight in closed spherical three-space as an eye walks forward along the same geodesic. Because light rays follow great circles on the 3-sphere, objects receding past a quarter turn reconverge toward the antipode of the eye instead of shrinking indefinitely, so an ellipsoid past the equator subtends a larger apparent outline than a nearer one.

Points of the 3-sphere are modelled as intrinsic antivectors, great spheres as extrinsic planes, and each ellipsoid as a dual quadric `Point <- Plane` placed by a motor `placements >> ellipsoid(placements << Plane)` and inverted into its primal polarity map `surfaces = bodies.inverse()`. From an eye point `eye = eye_frame >> origin`, the tangent outline cone `polar * (Point & polar) - surfaces * (eye & polar)` pulls back onto the equatorial screen `screen = eye_frame >> ScreenPoint` as a screen conic `(screen & normalized(screen)) - polar * polar` whose larger quadratic root `-polar(pixels) + (-conic(pixels, pixels)).square_root()` gives each hit's cotangent depth along its great circle.

In homogeneous $4\times 4$ projective matrix notation on $S^3 \subset \mathbb{R}^4$, placing an ellipsoid matrix $Q_0 = \operatorname{diag}(\tan^2\alpha_x, \tan^2\alpha_y, \tan^2\alpha_z, -1)^{-1}$ by separate $\mathrm{SO}(4)$ rotation blocks $R = R_{xw}R_{yw}R_{zw}R_{xy}$ gives $Q = R Q_0 R^{\mathsf T}$, projecting from camera position $\mathbf{e} = R_{\text{cam}}\mathbf{e}_4$ onto the tangent basis $S = R_{\text{cam}}[\mathbf{e}_1\;\mathbf{e}_2\;\mathbf{e}_3]$ forms the $3\times 3$ outline conic $C = (\mathbf{e}^{\mathsf T} Q\,\mathbf{e})^{-1} S^{\mathsf T} Q S - (\mathbf{e}^{\mathsf T} Q\,\mathbf{e})^{-2} S^{\mathsf T} Q\mathbf{e}\,\mathbf{e}^{\mathsf T} Q S$, and ray hits evaluate $\cot t = -(\mathbf{e}^{\mathsf T} Q\,\mathbf{e})^{-1}\mathbf{p}^{\mathsf T} S^{\mathsf T} Q\mathbf{e} + \sqrt{-\mathbf{p}^{\mathsf T} C\,\mathbf{p}}$.

In [ ]:
# In Colab, install numga and its example dependencies from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
from collections.abc import Iterator

%matplotlib inline
import numpy as np
from numga import Algebra, NumpyContext
from numga.gatype.traits import Versor

from examples.quadrics.s3_raytracer import render, scenarios
from examples.util.animation import save_figure, show_states

np.set_printoptions(precision=4, suppress=True)

ga = Algebra("x+y+z+w+")
ctx = NumpyContext(ga)
mv = ctx.multivector
Scalar = ga.gatype.scalar()                                                    # Scalar+
Point = ga.gatype.antivector()                                                 # Antivector-
Plane = ga.gatype.vector()                                                     # Vector+
# Directions from the origin correspond to duals of planes through it, spanned by x, y, and z.
ScreenPlane = Plane.derive.within(Algebra("x+y+z+"))                           # Vector+
ScreenPoint = ScreenPlane.derive.dual                                          # Antivector-
ScreenConic = ga.gatype((Scalar, ScreenPoint, ScreenPoint))                    # Scalar+ <- (ScreenPoint-, ScreenPoint-)
ScreenPolar = ga.gatype((Scalar, ScreenPoint))                                 # Scalar+ <- ScreenPoint-
Motor = ga.gatype.rotor()                                                      # Rotor+
# A primal quadric maps a point to its polar plane.
Quadric = ga.gatype((Plane, Point))                                            # Plane+ <- Point-


def unit(points: Point) -> Point:
    """Dividing `points` (`[...] Point-`) by their `w` weight and normalizing gives the positive-weight unit versor representatives."""
    return (points / (mv.w & points)).normalized().with_traits(Versor)         # [...] Point-


origin = unit(mv.w.dual())                                                     # [] Point-


def direction(coordinates: np.ndarray) -> Point:
    """Dualizing and normalizing `coordinates` (`[..., 3]`) gives unit equatorial points (`[...] Point-`) a quarter turn from `origin`."""
    return mv(ScreenPlane, coordinates).dual().normalized()                    # [...] Point-

## Identical ellipsoids along a geodesic

An ellipsoid centred at `origin` is built as a dual quadric summing its three equatorial axis dyads weighted by squared tangent half-widths and subtracting the origin dyad that closes the surface. Four copies are carried by motors to increasing geodesic distances along the forward axis, offset slightly sideways and vertically so none occludes another head-on, and inverted into primal quadrics `Plane <- Point`.

In [ ]:
n_bodies = scenarios.n_bodies
half_widths = scenarios.half_widths
distances = scenarios.distances
sideways, upward = scenarios.sideways, scenarios.upward
turn = scenarios.turn
colours = scenarios.colours
light = direction(scenarios.light_coordinates)                                 # [] Point-
field_of_view = scenarios.field_of_view
image_shape = scenarios.image_shape

In [ ]:
def placed_ellipsoids(
    half_widths: np.ndarray,
    distances: np.ndarray,
    sideways: np.ndarray,
    upward: np.ndarray,
    turn: float,
) -> Quadric:
    """Placing a central ellipsoid of angular `half_widths` (`[3]`) by geodesic offsets (`[n_bodies]`) and `turn` gives primal quadrics (`[n_bodies] Plane+ <- Point-`)."""
    # Principal directions a quarter turn from `origin` weighted by squared tangent half-widths minus the origin dyad close the dual ellipsoid.
    axes = direction(np.eye(3))                                                # [3] Point-
    ellipsoid = (axes * (axes & Plane) * np.tan(half_widths) ** 2).sum(axis=0) - origin * (origin & Plane)  # [] Point- <- Plane+
    to_distance = (mv.xw * (distances / 2)).exp()                              # [...] Motor+
    to_side = (mv.yw * (sideways / 2)).exp()                                   # [...] Motor+
    to_height = (mv.zw * (upward / 2)).exp()                                   # [...] Motor+
    spin = (mv.xy * (turn / 2)).exp()                                          # [] Motor+
    placements = to_distance * to_side * to_height * spin                      # [...] Motor+
    bodies = placements >> ellipsoid(placements << Plane)                      # [...] Point- <- Plane+
    return bodies.inverse()                                                    # [...] Plane+ <- Point-


surfaces = placed_ellipsoids(half_widths, distances, sideways, upward, turn)   # [n_bodies] Plane+ <- Point-
save_figure(
    render.walk_figure(mv.rotor(), surfaces, colours, light, field_of_view, image_shape),
    "s3_raytracer",
);

From nearest to farthest along the forward geodesic, the red (`0.7` rad) and green (`1.4` rad) ellipsoids shrink toward the equator at a quarter turn (`1.57` rad), whereas the blue (`2.1` rad) and yellow (`2.6` rad) ellipsoids lie past the equator where the camera's great circles reconverge, making the yellow ellipsoid appear larger than both the green and blue ones.

## Outlines and cotangent depths by projection

From the eye, the cone of rays tangent to each body equals the squared polar plane of the eye minus the body's primal quadric scaled by the eye's own value, which evaluates non-negative on every direction whose great circle meets the surface. Pulling the normalized quadric and the eye's polar plane onto the equatorial screen produces a screen conic and linear form whose larger root gives each covered pixel's cotangent depth.

In [ ]:
coarse_shape = scenarios.coarse_shape
n_coarse_pixels = coarse_shape[0] * coarse_shape[1]
check_stride = scenarios.check_stride

In [ ]:
def pixel_chart(field_of_view: float, shape: tuple[int, int]) -> ScreenPoint:
    """Pinhole grid coordinates of `shape` (`(n_rows, n_columns)`) and `field_of_view` form homogeneous screen points (`[n_rows * n_columns] ScreenPoint-`)."""
    n_rows, n_columns = shape
    vertical, horizontal = np.meshgrid(
        np.linspace(1, -1, n_rows) * n_rows / n_columns,
        np.linspace(-1, 1, n_columns),
        indexing="ij",
    )
    half_width = np.tan(field_of_view / 2)
    axes = mv(ScreenPlane, np.eye(3)).dual()                                   # [3] ScreenPoint-
    return axes[0] + axes[1] * (half_width * horizontal.ravel()) + axes[2] * (half_width * vertical.ravel())  # [...] ScreenPoint-


def outlines(eye_frame: Motor, surfaces: Quadric) -> Quadric:
    """Subtracting each quadric scaled by the eye's own value from the squared polar plane of the eye gives the tangent outline cones (`[...] Plane+ <- Point-`)."""
    eye = eye_frame >> origin                                                  # [] Point-
    polar = surfaces(eye)                                                      # [...] Plane+
    return polar * (Point & polar) - surfaces * (eye & polar)                  # [...] Plane+ <- Point-


chart = pixel_chart(field_of_view, coarse_shape)                               # [n_coarse_pixels] ScreenPoint-
cones = outlines(mv.rotor(), surfaces).reshape(-1, 1)                          # [n_bodies, 1] Plane+ <- Point-
covered = (chart & cones(chart)) >= 0                                          # [n_bodies, n_coarse_pixels]
eye_polars = surfaces.reshape(-1, 1)(origin)                                   # [n_bodies, 1] Plane+
ray_polars = surfaces.reshape(-1, 1)(chart)                                    # [n_bodies, n_coarse_pixels] Plane+
# Evaluating the quadric along the great circle `origin + chart * t` gives `quad_a * t**2 + 2 * quad_b * t + quad_c`.
quad_a = chart & ray_polars                                                    # [n_bodies, n_coarse_pixels] Scalar+
quad_b = chart & eye_polars                                                    # [n_bodies, n_coarse_pixels] Scalar+
quad_c = origin & eye_polars                                                   # [n_bodies, 1] Scalar+
discriminant = quad_b * quad_b - quad_a * quad_c                               # [n_bodies, n_coarse_pixels] Scalar+

In [ ]:
def project(eye_frame: Motor, surfaces: Quadric) -> tuple[ScreenConic, ScreenPolar]:
    """Pulling `surfaces` (`[...] Plane+ <- Point-`) back to the screen of `eye_frame` gives the outline conics and eye-polar linear forms."""
    eye = eye_frame >> origin                                                  # [] Point-
    screen = eye_frame >> ScreenPoint                                          # [] Point- <- ScreenPoint-
    normalized = surfaces / (eye & surfaces(eye))                              # [...] Plane+ <- Point-
    polar = screen & normalized(eye)                                           # [...] Scalar+ <- ScreenPoint-
    return (screen & normalized(screen)) - polar * polar, polar                # [...] Scalar+ <- (ScreenPoint-, ScreenPoint-), [...] Scalar+ <- ScreenPoint-


def reproject(conic: ScreenConic, polar: ScreenPolar, pixels: ScreenPoint) -> Scalar:
    """The larger root of the screen quadratic from `conic` and `polar` gives each pixel's first-hit cotangent depth (`[...] Scalar+`, `NaN` on misses)."""
    with np.errstate(invalid="ignore"):
        return -polar(pixels) + (-conic(pixels, pixels)).square_root()         # [...] Scalar+


conics, polars = project(mv.rotor(), surfaces)                                 # [n_bodies] Scalar+ <- (ScreenPoint-, ScreenPoint-), Scalar+ <- ScreenPoint-
print("covered coarse pixels per body (0.7, 1.4, 2.1, 2.6 rad):", covered.sum(axis=1))

## Walking along the line of sight

Advancing the eye frame along the forward geodesic while keeping the ellipsoids fixed sweeps each body's distance toward zero. Per pixel, the shader selects the body with the largest cotangent depth, reconstructs the hit on the 3-sphere, and shades its polar plane with spherical point-light falloff and great-circle shadow arcs.

In [ ]:
walk_distance = scenarios.walk_distance
n_frames = scenarios.n_frames
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
def walk_frames(walk_distance: float, n_frames: int) -> Iterator[Motor]:
    """Yields `n_frames` eye-frame motors (`[] Motor+`) advancing by `walk_distance` along the forward geodesic."""
    for step in np.linspace(0, walk_distance, n_frames, endpoint=False):
        yield (mv.xw * (step / 2)).exp()                                       # [] Motor+


recorded_frames = show_states(
    walk_frames(walk_distance, n_frames),
    render.walk_animation,
    surfaces,
    colours,
    light,
    field_of_view,
    image_shape,
    name="s3_raytracer",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
);

As the eye advances `1.2` radians forward, the red ellipsoid passes overhead, the green ellipsoid approaches close range, and the blue and yellow ellipsoids cross the equator from the far hemisphere so their outlines first narrow at a quarter turn before growing again.

In [ ]:
# checks
# Covered pixels from the farthest ellipsoid (`2.6` rad, past the equator) exceed those of the middle ones;
# the outline cone matches the great-circle discriminant away from grazing rays, the screen conic equals
# `-discriminant / quad_c**2`, reprojected hits lie on their surfaces, and the walk streamed `n_frames` motors.
assert len(recorded_frames) == n_frames
assert covered.sum(axis=1)[3] > covered.sum(axis=1)[1]
clear = discriminant.abs() > 1e-3 * discriminant.abs().max(axis=1, keepdims=True)
assert np.array_equal(covered[clear], (discriminant >= 0)[clear])
conic_residual = conics.reshape(-1, 1)(chart, chart) + discriminant / (quad_c * quad_c)  # [n_bodies, n_coarse_pixels] Scalar+
assert (conic_residual.abs() < 1e-6).all()
for body_index in range(n_bodies):
    hit_indices = np.nonzero(covered[body_index])[0][::check_stride]
    depths = reproject(conics[body_index], polars[body_index], chart[hit_indices])  # [...] Scalar+
    hits = (origin * depths + chart[hit_indices]).normalized()                 # [...] Point-
    assert ((hits & surfaces[body_index](hits)).abs() < 1e-6).all()